# 31장 · 반복 제어 · 평가 결과와 수정 횟수로 종료하기

수정 루프에는 평가 통과 조건과 최대 수정 횟수가 필요합니다. 평가를 먼저 하고 실패했을 때만 수정하면, 처음부터 기준을 통과한 답에는 수정 호출을 쓰지 않습니다.

## 수정 가능 횟수 읽기

`next_action`은 현재 판정, 이미 수정한 횟수, 허용 횟수를 받아 다음 행동을 반환합니다. 아직 통과하지 못했고 허용 횟수도 모두 쓴 경우에 어느 줄에서 함수가 끝나는지 확인하세요.

In [ ]:
def next_action(passed, edits, limit):
    if passed:
        return "완료"
    if edits >= limit:
        return "검토 요청"
    return "수정"

print(next_action(False, edits=1, limit=1))

검토 요청


이미 한 번 수정했으므로 다시 실패하면 “검토 요청”으로 끝납니다. passed만 True로 바꾸면 같은 횟수에서도 “완료”가 됩니다.

## 응용 실습 · 수정 횟수 제한하기

## 실습 준비

8장에서 모델을 호출했던 Jupyter 커널을 선택하고, 이 장의 실습 묶음을 풀어 노트북과 pre-mid 폴더를 나란히 둡니다. 새로운 커널이라면 빈 셀에서 `%pip install -U openai`를 실행하고 수업용 API 키를 환경 변수로 설정한 뒤 시작하세요. 키는 노트북 셀이나 제출물에 적지 않습니다. 아래 함수에는 매번 지시와 이번에 읽을 자료를 전달합니다. 출력은 실행할 때마다 표현이 달라질 수 있습니다.

In [ ]:
from pathlib import Path
import os
from openai import OpenAI

# ZIP을 푼 폴더에서 노트북을 열면 이 경로에 네 가지 대여 조항이 있습니다.
rules = Path("pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")
model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
question = '6월 10일에 A형을 직접 받았어요. 반납을 하루 미루려면 언제까지 어디에 남기면 되나요? 댓글만 쓰면 연장이 끝나는지도 궁금해요.'
judge_instruction = '규정·회원 문의·초안을 대조하세요. 회원이 물은 모든 행동에 필요한 조건이 있고\n규정과 일치하면 첫 줄에 통과, 아니면 수정이라고 쓰세요. 다음 줄에는\n원문 조건과 초안에서 빠지거나 잘못된 구절을 짝지어 한두 문장으로 설명하세요.\n문의에서 묻지 않은 주변 규정을 추가하라고 요구하지 마세요.'
revise_instruction = '회원에게 보낼 초안을 고치세요. 검토 의견을 규정과 대조하고, 맞는 조건은 유지하세요. 설명 없이 수정한 안내만 쓰세요.'

## 반복하기 전에 종료 순서를 그려 보기

초안이 오면 항상 먼저 확인합니다. 통과했다면 멈춥니다. 미흡해도 이미 지정한 수정 횟수를 다 썼다면 마지막 평가 의견을 남기고 멈춥니다. 둘 다 아니면 한 번 고칩니다. 최대 수정 1회에서 최초 검사와 수정 뒤 검사가 각각 한 번씩 있을 수 있습니다. “검사 횟수”와 “수정 횟수”를 같은 수로 세지 마세요.

## inspect와 improve의 반환값 구별하기

`inspect(draft)`는 평가 결과를 줄바꿈에서 나눠 `(판정, 이유)` 두 값을 돌려줍니다. `improve(draft, reason)`는 초안과 이유를 받아 고친 글 한 개를 돌려줍니다. 이 두 기능을 분리해야 재평가할 초안과 그 초안을 고친 이유가 뒤섞이지 않습니다.

평가는 첫 줄의 판정과 뒤의 이유를 나누어 반환합니다. partition은 첫 줄바꿈을 기준으로 텍스트를 앞쪽·구분자·뒤쪽으로 나눕니다. 예상한 단어가 나오지 않으면 성공으로 간주하지 않고 원 응답을 확인합니다. 수정 함수에는 바로 전 검토 의견과 초안이 들어갑니다.

`raw.strip().partition("\n")`은 첫 줄바꿈에서 한 번만 나눠 (앞, 구분자, 나머지)를 돌려줍니다. 가운데 값을 쓰지 않아 변수 이름을 `_`로 두었습니다. 첫 줄이 “수정”이어도 다음 줄에 이유가 없으면 이 평가를 다음 호출에 쓸 수 없어 ValueError가 납니다.

### 한 줄짜리 판정에 이유가 있는지 확인하기

In [ ]:
review_text = "수정"
verdict, separator, reason = review_text.partition("\n")
print(verdict)
print(repr(reason))

수정
''


이유가 비어 있어도 `partition` 자체는 정상 종료합니다. 그래서 이어지는 `inspect`에 빈 이유를 거르는 조건이 필요합니다.

In [ ]:
def inspect(draft):
    material = rules + "\n문의\n" + question + "\n초안\n" + draft
    raw = ask(judge_instruction, material)
    verdict, _, reason = raw.strip().partition("\n")
    if verdict.strip() not in ("통과", "수정") or not reason.strip():
        raise ValueError("평가 형식 확인: " + raw)
    return verdict.strip(), reason.strip()

def improve(draft, reason):
    material = rules + "\n" + question + "\n초안\n" + draft + "\n검토 의견\n" + reason
    return ask(revise_instruction, material)

## review_until에서 검사·수정 횟수 구분하기

`review_until(draft, check, rewrite, limit=1)`에서는 함수 이름을 괄호 없이 넘깁니다. `check(draft)`에서 실제 평가가 일어나고, 수정이 필요하며 기회가 남았을 때만 `rewrite(draft, reason)`을 부릅니다. `limit=0`이면 최초 평가는 하고 수정은 한 번도 하지 않습니다.

다음 함수는 검사할 함수와 수정할 함수를 인자로 받습니다. 평소에는 실제 SDK 호출 함수를 넘기고, 종료 규칙을 시험할 때는 결과를 알 수 있는 작은 시험 함수를 넘길 수 있습니다. 각 시도의 초안과 이유를 같은 기록에 넣습니다.

`review_until(draft, check, rewrite, limit=1)`은 평가 함수와 수정 함수를 인자로 받습니다. 먼저 평가하고 나서 수정할지 결정하므로 수정 1회 뒤에도 평가가 한 번 더 필요합니다. `limit=0`으로 호출해도 첫 평가는 수행합니다. 그 첫 판정이 “수정”이면 `rewrite`를 부르기 전에 “수정 상한”으로 끝납니다.

In [ ]:
def review_until(draft, check, rewrite, limit=1):
    attempts, history = 0, []
    while True:
        verdict, reason = check(draft)
        history.append({"수정 횟수": attempts, "초안": draft, "판정": verdict, "이유": reason})
        if verdict == "통과":
            return draft, "기준 충족", history
        if attempts >= limit:
            return draft, "수정 상한", history
        draft = rewrite(draft, reason)
        attempts += 1  # 실제로 수정한 뒤에만 한 번 더합니다.

In [ ]:
candidate = "6월 11일 정오까지 신청 게시글에 연장 요청 댓글을 남겨 주세요."
final, stop, history = review_until(candidate, inspect, improve)
print(stop, history[-1]["수정 횟수"])
print(history[-1]["이유"])
print(final)

실제 판정이 잘못되었을 수도 있습니다. 통과라면 R4의 승인 주체가 최종 답에 있는지 직접 확인하세요. 상한이라면 마지막 평가 의견을 남겨 다음 담당자가 수동 검토할 수 있어야 합니다. 상한에 도달했다는 말은 답이 정확하다는 뜻이 아닙니다.

### 실습 · 수정 한도가 0이면 평가는 몇 번 일어날까요?

앞에서 만든 review_until에 테스트용 판정 함수와 수정 함수를 전달하세요. 판정이 “수정”이어도 limit=0에서 초안이 보존되는지, 기록은 몇 개인지 확인하세요.

In [ ]:
def check_short(text):
    return "수정", "한 문장 더 필요"

# limit=0으로 실행한 뒤 초안, 종료 이유, 기록 건수를 출력하세요.

<details><summary>정답 코드와 실행 결과</summary><pre><code>def check_short(text):
    return &quot;수정&quot;, &quot;한 문장 더 필요&quot;

final, stop, history = review_until(&quot;안녕&quot;, check_short, lambda text, reason: text + &quot;!&quot;, limit=0)
print(final, stop, len(history))</code></pre><pre>안녕 수정 상한 1</pre><p>최초 평가 한 건은 남습니다. 수정 함수가 호출되었다면 느낌표가 붙었을 텐데 최종 글은 처음과 같습니다.</p></details>

## 운 좋게 통과해도 상한 경로 확인하기

아래 검사는 모델 없이 종료 동작만 살핍니다. “통과”와 “수정”을 고정해 반환하는 시험 함수를 각각 한 번 사용합니다. 통과 경로에서는 수정 호출 자체가 일어나지 않아야 하고, 항상 미흡한 예에서는 처음과 마지막에 각각 미흡 판정이 남아야 합니다.

In [ ]:
def already_ok(draft):
    return "통과", "시험 입력에 합의한 조건이 있음"

def always_missing(draft):
    return "수정", "시험 입력에 승인 조건을 계속 누락시킴"

def keep_draft(draft, reason):
    return draft  # 상한에서 멈추는지 보기 위해 일부러 고치지 않습니다.

print(review_until(candidate, already_ok, keep_draft)[1:])
stuck, why, trail = review_until(candidate, always_missing, keep_draft)
print(why, "수정 횟수", trail[-1]["수정 횟수"], "평가 횟수", len(trail))

출력에서 두 번째 경로는 수정 1회, 평가 2회입니다. 실제 모델에게 계속 틀리라고 요구해 같은 결과를 얻는 것보다 상한 규칙 자체를 직접 확인할 수 있습니다. 실제 평가의 품질은 앞 셀처럼 규정과 대조합니다.

## 초안이 한 번 바뀐 뒤 어디서 멈출까

```mermaid
flowchart TD
    A["초안 검사·기록"] --> B{"통과했나?"}
    B -->|예| C["기준 충족"]
    B -->|아니요| D{"수정 상한 도달?"}
    D -->|예| E["미해결 사유 보관"]
    D -->|아니요| F["의견으로 수정"]
    F --> G["수정 횟수 증가"]
    G --> A
```

상한이 1이고 첫 답에 승인 조건이 빠졌다면 첫 검사 뒤 F로 갑니다. 고친 뒤 A로 돌아와 다시 검사했을 때도 빠졌다면 E에서 멈춥니다. 처음부터 조건이 맞았던 문의는 첫 검사 후 C로 가며 수정 호출은 0회입니다.

## 세 가지 종료 기록을 나란히 읽기

**기대 동작 예시**

| 최초 판정 | 수정 뒤 판정 | 모델 평가 수 | 수정 수 | 종료 |
|---|---|---:|---:|---|
| 통과 | 해당 없음 | 1 | 0 | 기준 충족 |
| 수정 | 통과 | 2 | 1 | 기준 충족 |
| 수정 | 수정 | 2 | 1 | 수정 상한 |

마지막 행의 답도 문자열로는 완성되어 보입니다. 해결하지 못한 조건과 마지막 평가 의견까지 담당자에게 넘겨야 검토할 수 있습니다. 횟수 상한을 올리기 전에 어떤 조건이 반복해서 누락되는지 먼저 읽으세요.

## 초안과 마지막 판정을 따로 보관하는 이유

첫 작성에서 이미 승인 조건이 맞고 시각만 틀렸습니다. 수정한 답은 시각을 고쳤지만 이번에는 승인 주체를 지웠습니다. 최신 답만 남았다면 무엇이 좋아지고 나빠졌는지 알기 어렵습니다. 학생 A는 최초 초안, 학생 B는 수정본에서 각각 “언제”, “어디에”, “누가 확정” 세 표현을 색으로 표시한 뒤 종이를 교환하세요. 수정 횟수가 1회여도 회원에게 바로 보낼지, 담당자가 다시 볼지 서로 다르게 판단할 수 있습니다. 왜 마지막 평가가 필요한지 여기서 확인합니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 한 번에 통과할 답과 두 번째 기회가 필요한 답

개인 실습 90분. 원문을 읽고 빠짐없는 답을 하나 쓰고, 시각과 승인 둘 중 하나가 틀린 답을 따로 준비합니다. 두 입력을 실제 평가·수정에 보내 예상한 횟수와 실측 횟수를 대조하세요. limit=0인 상태에서 미흡한 답을 시험 함수로 확인한 뒤, 사람에게 넘길 “아직 고쳐야 할 문장”을 한 줄로 남기세요.

In [ ]:
good = "6월 11일 정오까지 원래 신청 게시글에 하루 연장을 요청하는 댓글을 남기세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다."
# good에서 승인 문장을 지운 버전도 별도로 시험하세요.
# 두 입력을 각각 review_until에 넣고 반환된 종료 사유와 마지막 기록의 수정 횟수를 비교하세요.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>good = &quot;6월 11일 정오까지 원래 신청 게시글에 하루 연장 요청 댓글을 남기세요. 담당자 승인 댓글이 달려야 반납일이 바뀝니다.&quot;
missing = &quot;6월 11일 정오까지 원래 신청 게시글에 하루 연장 요청 댓글을 남기세요.&quot;
for candidate in (good, missing):
    last_answer, stop_reason, attempts = review_until(candidate, inspect, improve)
    print(stop_reason, attempts[-1][&quot;수정 횟수&quot;], last_answer)
print(review_until(missing, always_missing, keep_draft, limit=0)[1:])</code></pre>
<p>사람의 판정에서 good은 바로 통과할 수 있습니다. 실제 모델이 다르게 판정했다면 평가자가 요구한 조건을 원문에서 확인합니다. limit=0에서 시험용 미흡 판정은 수정 0회, 평가 1회, 종료 사유 수정 상한이 맞습니다.</p>
</details>

## 다음 작업에 가져갈 기록

처음과 마지막 초안, 검토 의견, 종료 사유를 남겼습니다. 다음 장에서는 이 기록을 챗봇 화면에 붙여 회원이 읽을 답과 담당자가 점검할 과정을 함께 보여 줍니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)